In [ ]:
# Original pyBuildingEnergy — official PyPI release
!pip install -q pybuildingenergy

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Fork (AIB) source — read directly from this repository ──────────────────
# The fork source lives inside pyBuildingEnergy/ in this repo.
# Reading it directly means the notebook always reflects the live local source
# without any GitHub clone step. Update the fork code in the repo and re-run
# from Cell 1 to pick up changes automatically.
GITHUB_REPO_URL = "https://github.com/samiraghafarigousheh-sys/PyBuildingEnergy_AIBteam_AU.git"
GITHUB_BRANCH   = "claude/energy-calc-comparison-7ujmt2"  # switch to "main" once the fix PR is merged

_nb_dir = os.path.abspath(os.getcwd())          # works in Colab and locally
FORK_SRC   = os.path.join(_nb_dir, "pyBuildingEnergy", "src")
FORK_TESTS = os.path.join(_nb_dir, "pyBuildingEnergy", "tests")

if not os.path.isdir(FORK_SRC):
    # Repo isn't checked out in this runtime (e.g. a fresh Colab session) — clone it.
    import subprocess
    subprocess.run(
        ["git", "clone", "--branch", GITHUB_BRANCH, "--depth", "1", GITHUB_REPO_URL],
        check=True,
    )
    _repo_dir  = os.path.join(_nb_dir, "PyBuildingEnergy_AIBteam_AU")
    FORK_SRC   = os.path.join(_repo_dir, "pyBuildingEnergy", "src")
    FORK_TESTS = os.path.join(_repo_dir, "pyBuildingEnergy", "tests")

assert os.path.isdir(FORK_SRC),   f"Fork source not found: {FORK_SRC}"
assert os.path.isdir(FORK_TESTS), f"Fork tests not found: {FORK_TESTS}"

WEATHER_SOURCE = "pvgis"   # same weather source used for ALL THREE engines:
# Engine 1 & 2 (ISO 52016): PVGIS TMY fetched internally via WEATHER_SOURCE
# Engine 3 (EnergyPlus):    PVGIS TMY downloaded as EPW in Cell 11 (same lat/lon)
# Coordinates: lat=-37.800, lon=144.968 (Carlton, Melbourne) — identical across engines


In [ ]:
def build_bui():
    """Returns a fresh copy of the Apt 305 building dictionary (no shared state)."""

    # --- Construction U-values & thermal capacity — Australian BCA 2006 minimum-spec ---
    U_EXT_WALL  = 1.00   # brick veneer / precast w/ R1.0 insulation
    U_INT_WALL  = 2.50   # concrete block + plasterboard, no insulation
    U_INT_SLAB  = 1.80   # 200 mm concrete intermediate floor
    U_WINDOW    = 5.40   # aluminium-frame single glazing
    G_WINDOW    = 0.65   # SHGC of clear single glazing

    ABS_EXT_WALL = 0.75  # dark red brick
    ABS_INT      = 0.0

    C_EXT_WALL = 0          # thermal mass zeroed for engine-method-only comparison
    C_INT_WALL = 0          # thermal mass zeroed for engine-method-only comparison
    C_INT_SLAB = 0          # thermal mass zeroed for engine-method-only comparison
    C_WINDOW   = 0

    # --- Geometry ---
    LEN_NS  = 5.0   # N-S length, m (west facade width, along Barry St)
    LEN_EW  = 4.0   # E-W depth, m
    HEIGHT  = 2.7   # ceiling height, m

    FLOOR_AREA = LEN_NS * LEN_EW
    VOLUME     = FLOOR_AREA * HEIGHT

    A_WEST_GROSS  = LEN_NS * HEIGHT
    A_EAST_GROSS  = LEN_NS * HEIGHT
    A_NORTH_GROSS = LEN_EW * HEIGHT
    A_SOUTH_GROSS = LEN_EW * HEIGHT

    WIN_WIDTH_FIXED,    WIN_HEIGHT_FIXED    = 0.9, 0.9
    WIN_WIDTH_OPERABLE, WIN_HEIGHT_OPERABLE = 0.9, 0.9
    A_WINDOW_FIXED    = WIN_WIDTH_FIXED * WIN_HEIGHT_FIXED
    A_WINDOW_OPERABLE = WIN_WIDTH_OPERABLE * WIN_HEIGHT_OPERABLE
    A_WINDOW_TOTAL    = A_WINDOW_FIXED + A_WINDOW_OPERABLE
    A_WEST_OPAQUE     = A_WEST_GROSS - A_WINDOW_TOTAL

    bui = {
        "building": {
            "name": "Apt_305_50_Barry_St_Carlton",
            "azimuth_relative_to_true_north": 0,
            "latitude":  -37.800,
            "longitude": 144.968,
            "exposed_perimeter": 0,
            "height": HEIGHT,
            "wall_thickness": 0.20,
            "n_floors": 1,
            "building_type_class": "Residential_apartment",
            "adj_zones_present": True,
            "number_adj_zone": 5,
            "net_floor_area": FLOOR_AREA,
            "construction_class": "class_iii",
            "construction_year": "2006-today",
            "country": "Australia",
        },
        "adjacent_zones": [
            {
                "name": "apt_above",
                "orientation_zone": {"azimuth": 270.0},
                "area_facade_elements": np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_EXT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": VOLUME,
                "building_type_class": "Residential_apartment",
                "a_use": FLOOR_AREA,
            },
            {
                "name": "apt_below",
                "orientation_zone": {"azimuth": 270.0},
                "area_facade_elements": np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_EXT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": VOLUME,
                "building_type_class": "Residential_apartment",
                "a_use": FLOOR_AREA,
            },
            {
                "name": "apt_north",
                "orientation_zone": {"azimuth": 0.0},
                "area_facade_elements": np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": VOLUME,
                "building_type_class": "Residential_apartment",
                "a_use": FLOOR_AREA,
            },
            {
                "name": "apt_south",
                "orientation_zone": {"azimuth": 180.0},
                "area_facade_elements": np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": VOLUME,
                "building_type_class": "Residential_apartment",
                "a_use": FLOOR_AREA,
            },
            {
                "name": "corridor",
                "orientation_zone": {"azimuth": 90.0},
                "area_facade_elements": np.array([81.0, 5.4, 81.0, 5.4, 60.0, 60.0]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_INT_WALL] * 6),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": 162.0,
                "building_type_class": "Residential_apartment",
                "a_use": 60.0,
            },
        ],
        "building_surface": [
            {
                "name": "West exterior wall (opaque)", "type": "opaque", "area": A_WEST_OPAQUE,
                "sky_view_factor": 0.5, "u_value": U_EXT_WALL, "solar_absorptance": ABS_EXT_WALL,
                "thermal_capacity": C_EXT_WALL, "orientation": {"azimuth": 270.0, "tilt": 90.0},
                "name_adj_zone": None, "height": HEIGHT, "length": LEN_NS,
            },
            {
                "name": "North wall to Apt 306", "type": "opaque", "area": A_NORTH_GROSS,
                "sky_view_factor": 0.0, "u_value": U_INT_WALL, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_WALL, "orientation": {"azimuth": 0.0, "tilt": 90.0},
                "name_adj_zone": "apt_north", "height": HEIGHT, "length": LEN_EW,
            },
            {
                "name": "South wall to Apt 304", "type": "opaque", "area": A_SOUTH_GROSS,
                "sky_view_factor": 0.0, "u_value": U_INT_WALL, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_WALL, "orientation": {"azimuth": 180.0, "tilt": 90.0},
                "name_adj_zone": "apt_south", "height": HEIGHT, "length": LEN_EW,
            },
            {
                "name": "East wall to corridor", "type": "opaque", "area": A_EAST_GROSS,
                "sky_view_factor": 0.0, "u_value": U_INT_WALL, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_WALL, "orientation": {"azimuth": 90.0, "tilt": 90.0},
                "name_adj_zone": "corridor", "height": HEIGHT, "length": LEN_NS,
            },
            {
                "name": "Floor to Apt 205", "type": "opaque", "area": FLOOR_AREA,
                "sky_view_factor": 0.0, "u_value": U_INT_SLAB, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_SLAB, "orientation": {"azimuth": 0.0, "tilt": 0.0},
                "name_adj_zone": "apt_below", "height": LEN_NS, "length": LEN_EW,
            },
            {
                "name": "Ceiling to Apt 405", "type": "opaque", "area": FLOOR_AREA,
                "sky_view_factor": 0.0, "u_value": U_INT_SLAB, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_SLAB, "orientation": {"azimuth": 0.0, "tilt": 0.0},
                "name_adj_zone": "apt_above", "height": LEN_NS, "length": LEN_EW,
            },
            {
                "name": "West window — fixed", "type": "transparent", "area": A_WINDOW_FIXED,
                "sky_view_factor": 0.5, "u_value": U_WINDOW, "solar_absorptance": 0.5,
                "thermal_capacity": C_WINDOW, "orientation": {"azimuth": 270.0, "tilt": 90.0},
                "name_adj_zone": None, "height": WIN_HEIGHT_FIXED, "g_value": G_WINDOW,
                "width": WIN_WIDTH_FIXED, "parapet": 1.0, "shading": True,
                "shading_type": "horizontal_overhang", "width_or_distance_of_shading_elements": 0.05,
                "overhang_proprieties": {"width_of_horizontal_overhangs": 0.25},
            },
            {
                "name": "West window — operable", "type": "transparent", "area": A_WINDOW_OPERABLE,
                "sky_view_factor": 0.5, "u_value": U_WINDOW, "solar_absorptance": 0.5,
                "thermal_capacity": C_WINDOW, "orientation": {"azimuth": 270.0, "tilt": 90.0},
                "name_adj_zone": None, "height": WIN_HEIGHT_OPERABLE, "g_value": G_WINDOW,
                "width": WIN_WIDTH_OPERABLE, "parapet": 1.0, "shading": True,
                "shading_type": "horizontal_overhang", "width_or_distance_of_shading_elements": 0.05,
                "overhang_proprieties": {"width_of_horizontal_overhangs": 0.25},
            },
        ],
        "units": {
            "area": "m\u00b2", "u_value": "W/m\u00b2K", "thermal_capacity": "J/m\u00b2K",
            "azimuth": "degrees (0=N, 90=E, 180=S, 270=W)", "tilt": "degrees (0=horizontal, 90=vertical)",
            "internal_gain": "W/m\u00b2", "HVAC_profile": "0: off, 1: on",
        },
        "building_parameters": {
            "temperature_setpoints": {
                # Fixed setpoints, as used in copy_of_pybuildingenergy_my_house.ipynb.
                # The modified-engine cell overrides these with NCC zone-derived setpoints.
                "heating_setpoint": 18.0,
                "heating_setback":  15.0,
                "cooling_setpoint": 26.0,
                "cooling_setback":  28.0,
                "units": "\u00b0C",
            },
            "system_capacities": {
                "heating_capacity": 10_000_000.0,
                "cooling_capacity": 10_000_000.0,
                "units": "W",
            },
            "ventilation": {
                "ventilation_type": "occupancy",
                "flow_rate_per_person": 2.0,
                "units": "l/(s m\u00b2)",
                "custom_heat_transfer_coefficient_ventilation": None,
            },
            "internal_gains": [
                {
                    "name": "occupants", "full_load": 8.0,  # 2 people × 80 W/person sensible = 160 W / 20 m² = 8 W/m²
                    "weekday": [1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.4,0.5,0.5,0.5,0.4,0.5,0.5,0.5,0.5,0.5,1.0,1.0,1.0,1.0],
                    "weekend": [1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.8,0.7,0.7,0.7,0.7,0.5,0.5,0.7,0.8,1.0,1.0,1.0,1.0,1.0,1.0],
                },
                {
                    "name": "appliances", "full_load": 5.0,   # reduced from 25 to 5 W/m²
                    "weekday": [0.1,0.1,0.1,0.1,0.1,0.1,0.2,0.3,0.2,0.2,0.2,0.2,0.3,0.2,0.2,0.2,0.2,0.3,0.3,0.4,1.0,0.6,0.4,0.2],
                    "weekend": [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.2,0.3,0.4,0.3,0.3,0.4,0.3,0.3,0.3,0.3,0.4,0.4,0.5,1.0,0.6,0.4,0.2],
                },
                {
                    "name": "lighting", "full_load": 3.0,
                    "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.3,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.5,0.8,0.8,0.8,0.7,0.4,0.1],
                    "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.3,0.3,0.2,0.2,0.2,0.2,0.2,0.2,0.3,0.5,0.8,0.8,0.8,0.7,0.4,0.1],
                },
            ],
            "construction": {
                "wall_thickness": 0.20,
                "thermal_bridges": 1.5,
                "units": "m (thickness), W/mK (thermal bridges)",
            },
            "climate_parameters": {"coldest_month": 7, "units": "1-12 (January-December)"},
            "heating_profile": {
                "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0],
                "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0],
            },
            "cooling_profile": {
                "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0],
                "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0],
            },
            "ventilation_profile": {
                "weekday": [1.0] * 24,
                "weekend": [1.0] * 24,
            },
        },
    }
    return bui

FLOOR_AREA = build_bui()["building"]["net_floor_area"]
print(f"Floor area: {FLOOR_AREA} m2")


In [ ]:
from pybuildingenergy.source.check_input import sanitize_and_validate_BUI as orig_sanitize
from pybuildingenergy.source.utils import ISO52016 as OrigISO52016

bui_original = build_bui()
bui_original_fixed, report_original = orig_sanitize(bui_original, fix=True)

print("=== Validation (original engine) ===")
for r in report_original:
    fix_tag = " (fix applied)" if r["fix_applied"] else ""
    print(f"  [{r['level']}] {r['path']}: {r['msg']}{fix_tag}")

errors_original = [r for r in report_original if r["level"] == "ERROR"]
if errors_original:
    raise ValueError("Building input invalid for the original engine — fix errors above.")



print(f"=== Running ORIGINAL engine (weather: {WEATHER_SOURCE}) ===")
# The PyPI release returns a 2-tuple (hourly_sim, annual_results_df); the fork's engine
# (used below) returns a 3-tuple that also includes Sankey flow data. Handle both.
_orig_out = OrigISO52016.Temperature_and_Energy_needs_calculation(
    bui_original_fixed, weather_source=WEATHER_SOURCE
)
if len(_orig_out) == 3:
    hourly_original, annual_original, sankey_original = _orig_out
else:
    hourly_original, annual_original = _orig_out
    sankey_original = {}

Q_HC_original = hourly_original["Q_HC"]
heating_kWh_original = Q_HC_original[Q_HC_original > 0].sum() / 1000.0
cooling_kWh_original = -Q_HC_original[Q_HC_original < 0].sum() / 1000.0

print(f"\nORIGINAL engine — Heating need: {heating_kWh_original:8.1f} kWh/yr "
      f"({heating_kWh_original/FLOOR_AREA:5.1f} kWh/m2/yr)")
print(f"ORIGINAL engine — Cooling need: {cooling_kWh_original:8.1f} kWh/yr "
      f"({cooling_kWh_original/FLOOR_AREA:5.1f} kWh/m2/yr)")

In [ ]:
# Drop the PyPI package's cached modules so "pybuildingenergy" re-resolves to the fork
for mod_name in list(sys.modules):
    if mod_name == "pybuildingenergy" or mod_name.startswith("pybuildingenergy."):
        del sys.modules[mod_name]

# FORK_SRC and FORK_TESTS already set in Cell 1 from the local repo
for p in (FORK_SRC, FORK_TESTS):
    if p not in sys.path:
        sys.path.insert(0, p)

from pybuildingenergy.source.check_input import sanitize_and_validate_BUI as mod_sanitize
from pybuildingenergy.source.utils import ISO52016 as ModISO52016
from climate_setpoints import get_ncc_setpoints, apply_setpoints_to_building

print("Modified engine imported from:", ModISO52016.__module__)



bui_modified = build_bui()

# ── FIX 1: Use IDENTICAL setpoints across all three engines for a fair comparison.
# The fork's apply_setpoints_to_building() would overwrite these with NCC Zone 6
# values (20°C heating, 17°C setback, 24°C cooling/bedroom, 27°C cooling setback),
# which differ from the 18/15/26/28 setpoints used by Engine 1 and Engine 3.
# We still call get_ncc_setpoints for reference/logging, but do NOT apply them.
ncc_setpoints = get_ncc_setpoints(
    lat=bui_modified["building"]["latitude"],
    lon=bui_modified["building"]["longitude"],
)
print("=== NCC setpoints (for reference only — NOT applied to Engine 2) ===")
print(f"  NCC zone        : {ncc_setpoints['ncc_zone']}")
print(f"  NCC Heating setpoint: {ncc_setpoints['heating_setpoint']} C")
print(f"  NCC Cooling (bedroom): {ncc_setpoints['cooling_setpoint_bedroom']} C")
print(f"  NCC Cooling (living) : {ncc_setpoints['cooling_setpoint_living']} C")
print(f"  NCC Heating setback  : {ncc_setpoints['heating_setback']} C")
print(f"  NCC Cooling setback  : {ncc_setpoints['cooling_setback']} C")

# Explicitly force Engine 2 to the same setpoints as Engine 1 and Engine 3.
# These values match the temperature_setpoints block in build_bui().
bui_modified["building_parameters"]["temperature_setpoints"].update({
    "heating_setpoint": 18.0,
    "heating_setback":  15.0,
    "cooling_setpoint": 26.0,
    "cooling_setback":  28.0,
})
print("\n=== Engine 2 setpoints (overridden to match Engine 1 & 3) ===")
print(f"  heating_setpoint : {bui_modified['building_parameters']['temperature_setpoints']['heating_setpoint']} C")
print(f"  heating_setback  : {bui_modified['building_parameters']['temperature_setpoints']['heating_setback']} C")
print(f"  cooling_setpoint : {bui_modified['building_parameters']['temperature_setpoints']['cooling_setpoint']} C")
print(f"  cooling_setback  : {bui_modified['building_parameters']['temperature_setpoints']['cooling_setback']} C")



bui_modified_fixed, report_modified = mod_sanitize(bui_modified, fix=True)

print("=== Validation (modified engine) ===")
for r in report_modified:
    fix_tag = " (fix applied)" if r["fix_applied"] else ""
    print(f"  [{r['level']}] {r['path']}: {r['msg']}{fix_tag}")

errors_modified = [r for r in report_modified if r["level"] == "ERROR"]
if errors_modified:
    raise ValueError("Building input invalid for the modified engine — fix errors above.")

In [ ]:
print(f"=== Running MODIFIED engine (weather: {WEATHER_SOURCE}) ===")
_mod_out = ModISO52016.Temperature_and_Energy_needs_calculation(
    bui_modified_fixed, weather_source=WEATHER_SOURCE
)
if len(_mod_out) == 3:
    hourly_modified, annual_modified, sankey_modified = _mod_out
else:
    hourly_modified, annual_modified = _mod_out
    sankey_modified = {}

Q_HC_modified = hourly_modified["Q_HC"]
heating_kWh_modified = Q_HC_modified[Q_HC_modified > 0].sum() / 1000.0
cooling_kWh_modified = -Q_HC_modified[Q_HC_modified < 0].sum() / 1000.0

print(f"\nMODIFIED engine — Heating need: {heating_kWh_modified:8.1f} kWh/yr "
      f"({heating_kWh_modified/FLOOR_AREA:5.1f} kWh/m2/yr)")
print(f"MODIFIED engine — Cooling need: {cooling_kWh_modified:8.1f} kWh/yr "
      f"({cooling_kWh_modified/FLOOR_AREA:5.1f} kWh/m2/yr)")


In [ ]:
comparison_df = pd.DataFrame([
    {
        "engine": "Original pyBuildingEnergy (PyPI)",
        "setpoints": "Fixed 18C / 26C",
        "heating_kWh": round(heating_kWh_original, 1),
        "cooling_kWh": round(cooling_kWh_original, 1),
        "heating_kWh_m2": round(heating_kWh_original / FLOOR_AREA, 1),
        "cooling_kWh_m2": round(cooling_kWh_original / FLOOR_AREA, 1),
    },
    {
        "engine": "Modified AIB fork",
        "setpoints": "Fixed 18C / 26C (same as Engine 1 & 3 — NCC auto-derivation overridden)",
        "heating_kWh": round(heating_kWh_modified, 1),
        "cooling_kWh": round(cooling_kWh_modified, 1),
        "heating_kWh_m2": round(heating_kWh_modified / FLOOR_AREA, 1),
        "cooling_kWh_m2": round(cooling_kWh_modified / FLOOR_AREA, 1),
    },
])
comparison_df


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(len(comparison_df))
width = 0.35

ax.bar(x - width / 2, comparison_df["heating_kWh"], width, label="Heating need")
ax.bar(x + width / 2, comparison_df["cooling_kWh"], width, label="Cooling need")

ax.set_xticks(x)
ax.set_xticklabels(comparison_df["engine"], rotation=10, ha="right")
ax.set_ylabel("Annual energy need (kWh/yr)")
ax.set_title("Apt 305, 50 Barry St, Carlton\nOriginal vs Modified ISO 52016 engine")
ax.legend()
plt.tight_layout()
plt.show()



OUTPUT_PATH = "/content/engine_comparison_apt305.csv"
comparison_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")


## 6. What exactly is different between the two engines?

Summary of every concrete difference observed between the official PyPI `pybuildingenergy`
engine and the `Sarthak790/pybuildinenergy_AIB` fork, based on running both against the
*identical* Apt 305 building dictionary and weather source above.


In [ ]:
differences = [
    {
        "area": "Heating/cooling setpoints",
        "original_engine": "Fixed values supplied by the user (18C heating / 26C cooling, "
                            "from the BUI dictionary's temperature_setpoints block).",
        "modified_engine": "FIX APPLIED: Setpoints explicitly overridden to match Engine 1 & 3 "
                            "(18C heating / 15C setback / 26C cooling / 28C cooling setback). "
                            "The fork's NCC auto-derivation is called for reference only and "
                            "the result is NOT applied to the building dictionary.",
    },
    {
        "area": "Interior wall U-values (ISO 13370 ground coupling)",
        "original_engine": "Interior partition walls (sky_view_factor = 0, e.g. the walls "
                            "to Apt 306 / Apt 304 / the corridor) keep the U-value exactly "
                            "as supplied: 2.50 W/m2K.",
        "modified_engine": "FIX APPLIED (utils.py): The ISO 13370 basement-wall correction "
                            "now checks for name_adj_zone before applying; surfaces with a "
                            "named adjacent zone are interior partitions, not ground-contact "
                            "elements, so their U-values are left at the user-supplied value "
                            "(2.50 W/m2K for partition walls, 1.80 W/m2K for slabs). "
                            "The '[ISO 13370] Converted Basement Wall ...' log lines will no "
                            "longer appear for these surfaces.",
    },
    {
        "area": "Weather source used here",
        "original_engine": f"PVGIS TMY ({WEATHER_SOURCE}) — identical to modified engine in "
                            "this notebook for a fair comparison.",
        "modified_engine": f"PVGIS TMY ({WEATHER_SOURCE}) here; the fork's own test instead "
                            "hardcodes a local Sydney EPW file path by default.",
    },
    {
        "area": "Schedule injection",
        "original_engine": "Occupant/appliance/lighting schedules are read only from the "
                            "BUI dictionary's building_parameters.internal_gains block.",
        "modified_engine": "Same dictionary-based schedules work, but "
                            "Temperature_and_Energy_needs_calculation also accepts explicit "
                            "occupants_schedule_workdays/weekend, appliances_schedule_*, and "
                            "lighting_schedule_* keyword arguments that override the "
                            "dictionary values directly (used in the fork's sample_test.py).",
    },
    {
        "area": "Latent heat / humidity",
        "original_engine": "hourly_sim has no latent-heat or moisture columns — sensible "
                            "heating/cooling load (Q_HC) only.",
        "modified_engine": "hourly_sim additionally includes Q_Latent (latent load, Wh) and "
                            "x_air_in (indoor moisture content) columns — a humidity balance "
                            "the original engine does not compute.",
    },
    {
        "area": "Domestic hot water (DHW)",
        "original_engine": "Not part of Temperature_and_Energy_needs_calculation; would "
                            "need to be added manually with Volume_and_energy_DHW_calculation.",
        "modified_engine": "Same DHW function is available and is combined with Q_H/Q_C/"
                            "Q_Latent into a single annual total in the fork's test.",
    },
    {
        "area": "Return signature",
        "original_engine": "Returns a 2-tuple: (hourly_sim, annual_results_df).",
        "modified_engine": "Returns a 3-tuple: (hourly_sim, annual_results_df, sankey_data).",
    },
    {
        "area": "Validation messages (sanitize_and_validate_BUI)",
        "original_engine": "Warning text is partly in Italian, "
                            "e.g. \"exposed_perimeter era 0; impostato a 1.0\".",
        "modified_engine": "Equivalent warnings are in English, "
                            "e.g. \"exposed_perimeter was 0; set to 1.0\".",
    },
]

diff_df = pd.DataFrame(differences)
pd.set_option("display.max_colwidth", 100)
diff_df


---

## 7. EnergyPlus — Full Heat-Balance Simulation

> **Methodology note:** EnergyPlus uses a **full heat-balance engine** (CTF conduction + combined radiative–convective surface balances), whereas both ISO 52016 engines above use the **simplified 5R1C RC-network** from ISO 52016-1. Results will differ even with identical inputs — that methodological gap is itself a key finding of this comparison.

| Parameter | Choice made |
|---|---|
| Weather | PVGIS TMY for lat=−37.800, lon=144.968, downloaded as EPW (same source as ISO 52016 runs) |
| Adjacent zones | All 5 neighbours fixed at **20 °C** year-round (conditioned-neighbour approximation) |
| Setpoints | Heating **18 °C** / Cooling **26 °C** — same as the original engine |
| Thermal capacity | `Material:NoMass` used; mass is absent (mirroring ISO 52016-1 steady-state RC intent) |
| HVAC | `ZoneHVAC:IdealLoadsAirSystem` — unlimited capacity, matches pyBuildingEnergy assumption |
| EnergyPlus version | 24.1.0 (downloaded on first run, ~150 MB) |

In [ ]:
# ── EnergyPlus 24.1.0 — install via tar.gz (no interactive prompt) ───────────
import os, subprocess, sys, requests

EP_VERSION = "24.1.0"
EP_SHA     = "9d7789a3ac"
EP_DIR     = "/opt/energyplus"

if not os.path.exists(EP_DIR):
    tar_name = f"EnergyPlus-{EP_VERSION}-{EP_SHA}-Linux-Ubuntu22.04-x86_64.tar.gz"
    tar_url  = (
        f"https://github.com/NREL/EnergyPlus/releases/download/"
        f"v{EP_VERSION}/{tar_name}"
    )
    tar_path = f"/tmp/{tar_name}"

    print(f"Downloading EnergyPlus {EP_VERSION} (tar.gz) …")
    subprocess.run(["wget", "-q", "--show-progress", "-O", tar_path, tar_url], check=True)

    print("Extracting …")
    os.makedirs(EP_DIR, exist_ok=True)
    # --strip-components=1 removes the top-level folder inside the archive
    subprocess.run(
        ["tar", "-xzf", tar_path, "--strip-components=1", "-C", EP_DIR],
        check=True,
    )
    os.remove(tar_path)
    print(f"Installed → {EP_DIR}")
else:
    print(f"EnergyPlus already at {EP_DIR}")

# Confirm the binary exists before continuing
ep_bin = os.path.join(EP_DIR, "energyplus")
assert os.path.exists(ep_bin), f"energyplus binary not found in {EP_DIR}"
print(f"Binary OK: {ep_bin}")

if EP_DIR not in sys.path:
    sys.path.insert(0, EP_DIR)

# ── PVGIS TMY → EPW  (same weather source as the ISO 52016 simulations) ──────
EPW_PATH = "/content/Melbourne_PVGIS_TMY.epw"
if not os.path.exists(EPW_PATH):
    print("Downloading PVGIS TMY as EPW …")
    pvgis_url = (
        "https://re.jrc.ec.europa.eu/api/v5_2/tmy"
        "?lat=-37.800&lon=144.968&outputformat=epw"
    )
    r = requests.get(pvgis_url, timeout=120)
    r.raise_for_status()
    with open(EPW_PATH, "wb") as f:
        f.write(r.content)
    print(f"Saved EPW → {EPW_PATH}  ({len(r.content)//1024} kB)")
else:
    print(f"EPW already at {EPW_PATH}")

with open(EPW_PATH) as f:
    first = f.readline().strip()
print(f"EPW header: {first[:80]}")

In [ ]:
def _sched_frac(name, wd, we):
    rows = ["Schedule:Compact,\n  {},\n  Fraction,\n  Through: 12/31,".format(name),
            "  For: Weekdays,"]
    for h, v in enumerate(wd, 1):
        rows.append("  Until: {:02d}:00,{},".format(h, v))
    rows.append("  For: AllOtherDays,")   # covers weekends + all design-day types
    for h, v in enumerate(we, 1):
        rows.append("  Until: {:02d}:00,{}{}".format(h, v, "," if h < 24 else ";"))
    return "\n".join(rows)

def _sched_temp(name, wd, we):
    rows = ["Schedule:Compact,\n  {},\n  Temperature,\n  Through: 12/31,".format(name),
            "  For: Weekdays,"]
    for h, v in enumerate(wd, 1):
        rows.append("  Until: {:02d}:00,{},".format(h, v))
    rows.append("  For: AllOtherDays,")
    for h, v in enumerate(we, 1):
        rows.append("  Until: {:02d}:00,{}{}".format(h, v, "," if h < 24 else ";"))
    return "\n".join(rows)

def build_ep_idf(heating_sp=18.0, cooling_sp=26.0, heating_sb=15.0, cooling_sb=28.0):
    """
    EnergyPlus IDF for Apt 305 mirroring build_bui() inputs.
    Material:NoMass R-values back-calculated to match assembly U-values.
    5 neighbour surfaces fixed at 20 C via OtherSideCoefficients.
    ZoneHVAC:IdealLoadsAirSystem (unlimited capacity).
    Ventilation: 0.002 m3/s/m2.
    """
    bui  = build_bui()
    bp   = bui["building_parameters"]
    ig   = {g["name"]: g for g in bp["internal_gains"]}

    occ_wd = ig["occupants"]["weekday"];  occ_we = ig["occupants"]["weekend"]
    app_wd = ig["appliances"]["weekday"]; app_we = ig["appliances"]["weekend"]
    lit_wd = ig["lighting"]["weekday"];   lit_we = ig["lighting"]["weekend"]
    h_wd = bp["heating_profile"]["weekday"]; h_we = bp["heating_profile"]["weekend"]
    c_wd = bp["cooling_profile"]["weekday"]; c_we = bp["cooling_profile"]["weekend"]

    hsp_wd = [heating_sp if v > 0 else heating_sb for v in h_wd]
    hsp_we = [heating_sp if v > 0 else heating_sb for v in h_we]
    csp_wd = [cooling_sp if v > 0 else cooling_sb for v in c_wd]
    csp_we = [cooling_sp if v > 0 else cooling_sb for v in c_we]

    R_ext = round(max(0.001, 1/1.00 - 0.13 - 0.04), 4)
    R_int = round(max(0.001, 1/2.50 - 0.13 - 0.13), 4)
    R_slb = round(max(0.001, 1/1.80 - 0.13 - 0.13), 4)
    LEN_NS = 5.0; LEN_EW = 4.0; H = 2.7

    schedules = "\n\n".join([
        _sched_frac("Occ_Sched",  occ_wd, occ_we),
        _sched_frac("App_Sched",  app_wd, app_we),
        _sched_frac("Lit_Sched",  lit_wd, lit_we),
        _sched_temp("Heat_SP",    hsp_wd, hsp_we),
        _sched_temp("Cool_SP",    csp_wd, csp_we),
        "Schedule:Compact,\n  Always1,\n  Fraction,\n  Through: 12/31,"
        "\n  For: AllDays,\n  Until: 24:00,1.0;",
        "Schedule:Compact,\n  Activity_W,\n  Any Number,\n  Through: 12/31,"
        "\n  For: AllDays,\n  Until: 24:00,160.0;",
    ])

    parts = []

    parts.append(
        "  Version, 24.1;\n\n"
        "  SimulationControl, Yes, No, No, No, Yes;\n\n"
        "  Building,\n"
        "    Apt_305_50_Barry_St_Carlton,\n"
        "    0.0, Suburbs, 0.04, 0.004,\n"
        "    FullInteriorAndExterior, 25, 6;\n\n"
        "  Timestep, 6;\n\n"
        "  GlobalGeometryRules,\n"
        "    UpperLeftCorner, CounterClockWise, World;\n\n"
        "  Site:Location,\n"
        "    Melbourne_VIC_AUS,\n"
        "    -37.800, 144.968, 10.0, 31.0;\n\n"
        "  RunPeriod,\n"
        "    FullYear,\n"
        "    1, 1, , 12, 31, ,\n"
        "    Sunday, Yes, Yes, No, Yes, Yes;\n\n"
        "  ScheduleTypeLimits, Fraction, 0.0, 1.0, Continuous, Dimensionless;\n"
        "  ScheduleTypeLimits, Temperature, -100, 200, Continuous, Temperature;\n"
        "  ScheduleTypeLimits, Any Number, -1e10, 1e10, Continuous;"
    )

    parts.append(schedules)

    parts.append(
        "  Zone, Apt305, 0.0, 0.0, 0.0, 0.0, 1, 1, {}, {}, {};".format(
            H, LEN_EW*LEN_NS*H, LEN_EW*LEN_NS)
    )

    parts.append(
        "  Material:NoMass, Mat_ExtWall, MediumRough,  {}, 0.9, 0.75, 0.75;\n"
        "  Material:NoMass, Mat_IntWall, MediumSmooth, {}, 0.9, 0.0,  0.0;\n"
        "  Material:NoMass, Mat_IntSlab, MediumSmooth, {}, 0.9, 0.0,  0.0;\n"
        "  WindowMaterial:SimpleGlazingSystem, Mat_Window, 5.40, 0.65;\n\n"
        "  Construction, Con_ExtWall, Mat_ExtWall;\n"
        "  Construction, Con_IntWall, Mat_IntWall;\n"
        "  Construction, Con_IntSlab, Mat_IntSlab;\n"
        "  Construction, Con_Window,  Mat_Window;".format(R_ext, R_int, R_slb)
    )

    parts.append(
        "  SurfaceProperty:OtherSideCoefficients,\n"
        "    OSC_Adj21C,\n"
        "    8.0,\n"
        "    21.0,\n"
        "    1.0, 0.0, 0.0, 0.0, 0.0;"
    )

    parts.append(
        "  BuildingSurface:Detailed,\n"
        "    WestWall, Wall, Con_ExtWall, Apt305, ,\n"
        "    Outdoors, , SunExposed, WindExposed, , 4,\n"
        "    0.0, {LN}, {H},\n"
        "    0.0, {LN}, 0.0,\n"
        "    0.0, 0.0, 0.0,\n"
        "    0.0, 0.0, {H};\n\n"
        "  BuildingSurface:Detailed,\n"
        "    NorthWall, Wall, Con_IntWall, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj21C, NoSun, NoWind, , 4,\n"
        "    {LE}, {LN}, {H},\n"
        "    {LE}, {LN}, 0.0,\n"
        "    0.0, {LN}, 0.0,\n"
        "    0.0, {LN}, {H};\n\n"
        "  BuildingSurface:Detailed,\n"
        "    EastWall, Wall, Con_IntWall, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj21C, NoSun, NoWind, , 4,\n"
        "    {LE}, 0.0, {H},\n"
        "    {LE}, 0.0, 0.0,\n"
        "    {LE}, {LN}, 0.0,\n"
        "    {LE}, {LN}, {H};\n\n"
        "  BuildingSurface:Detailed,\n"
        "    SouthWall, Wall, Con_IntWall, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj21C, NoSun, NoWind, , 4,\n"
        "    0.0, 0.0, {H},\n"
        "    0.0, 0.0, 0.0,\n"
        "    {LE}, 0.0, 0.0,\n"
        "    {LE}, 0.0, {H};\n\n"
        "  BuildingSurface:Detailed,\n"
        "    FloorSlab, Floor, Con_IntSlab, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj21C, NoSun, NoWind, , 4,\n"
        "    0.0, 0.0, 0.0,\n"
        "    0.0, {LN}, 0.0,\n"
        "    {LE}, {LN}, 0.0,\n"
        "    {LE}, 0.0, 0.0;\n\n"
        "  BuildingSurface:Detailed,\n"
        "    CeilingSlab, Ceiling, Con_IntSlab, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj21C, NoSun, NoWind, , 4,\n"
        "    0.0, 0.0, {H},\n"
        "    {LE}, 0.0, {H},\n"
        "    {LE}, {LN}, {H},\n"
        "    0.0, {LN}, {H};\n\n"
        "  FenestrationSurface:Detailed,\n"
        "    WestWin_Fixed, Window, Con_Window, WestWall, , , , 1.0, 4,\n"
        "    0.0, {y1h}, 1.9,  0.0, {y1h}, 1.0,\n"
        "    0.0, {y1l}, 1.0,  0.0, {y1l}, 1.9;\n\n"
        "  FenestrationSurface:Detailed,\n"
        "    WestWin_Operable, Window, Con_Window, WestWall, , , , 1.0, 4,\n"
        "    0.0, {y2h}, 1.9,  0.0, {y2h}, 1.0,\n"
        "    0.0, {y2l}, 1.0,  0.0, {y2l}, 1.9;".format(
            H=H, LN=LEN_NS, LE=LEN_EW,
            y1h=LEN_NS-1.0, y1l=LEN_NS-1.9,
            y2h=LEN_NS-2.5, y2l=LEN_NS-3.4,
        )
    )

    parts.append(
        # People: Method, LightingLevel(blank), PeoplePerArea, ZoneAreaPerPerson(blank),
        #         FracRadiant, SensibleHeatFrac(blank), ActivitySchedule
        "  People,\n"
        "    Apt305_People, Apt305, Occ_Sched,\n"
        "    People/Area, , 0.1, , 0.3, , Activity_W;\n\n"
        # 0.1 people/m2 x 20 m2 = 2 people (matching ISO engines: 2 x 80 W sensible = 8 W/m2)
        # ElectricEquipment: Method, DesignLevel(blank), W/Area, W/Person(blank),
        #   FracLatent, FracRadiant, FracLost
        "  ElectricEquipment,\n"
        "    Apt305_Appliances, Apt305, App_Sched,\n"
        "    Watts/Area, , 5.0, , 0.0, 0.5, 0.0;\n\n"
        # Lights: Method, LightingLevel(blank), W/Area, W/Person(blank),
        #   ReturnAirFrac, FracRadiant, FracVisible
        "  Lights,\n"
        "    Apt305_Lights, Apt305, Lit_Sched,\n"
        "    Watts/Area, , 3.0, , 0.0, 0.72, 0.18;"
    )

    parts.append(
        "  DesignSpecification:OutdoorAir,\n"
        "    Apt305_OA, Flow/Area, 0.002, , , , Always1;"
    )

    parts.append(
        "  ZoneHVAC:IdealLoadsAirSystem,\n"
        "    Apt305_IdealLoads, ,\n"
        "    Apt305_SupplyAir, Apt305_ExhaustAir, ,\n"
        "    50, 12, 0.010, 0.010,\n"
        "    NoLimit, , , NoLimit, , ,\n"
        "    , ,\n"
        "    ConstantSupplyHumidityRatio, ,\n"
        "    ConstantSupplyHumidityRatio,\n"
        "    Apt305_OA, , , , , , ;\n\n"
        "  ZoneHVAC:EquipmentList,\n"
        "    Apt305_EqpList,\n"
        "    SequentialLoad,\n"
        "    ZoneHVAC:IdealLoadsAirSystem, Apt305_IdealLoads, 1, 1;\n\n"
        "  ZoneHVAC:EquipmentConnections,\n"
        "    Apt305, Apt305_EqpList,\n"
        "    Apt305_SupplyAir, Apt305_ExhaustAir,\n"
        "    Apt305_ZoneAirNode;\n\n"
        "  NodeList, Apt305_SupplyAir, Apt305_SupplyAir_Node;"
    )

    parts.append(
        "  ThermostatSetpoint:DualSetpoint, Apt305_DualSP, Heat_SP, Cool_SP;\n\n"
        "  ZoneControl:Thermostat,\n"
        "    Apt305_Thermostat, Apt305, Always1,\n"
        "    ThermostatSetpoint:DualSetpoint, Apt305_DualSP;"
    )

    parts.append(
        "  OutputControl:Table:Style, HTML;\n"
        "  Output:Table:SummaryReports, AllSummary;\n"
        "  Output:Variable, Apt305, Zone Ideal Loads Heating Energy, Hourly;\n"
        "  Output:Variable, Apt305, Zone Ideal Loads Cooling Energy, Hourly;\n"
        "  Output:Diagnostics, DisplayAllWarnings;"
    )

    return "\n\n".join(parts)


In [ ]:
import glob, os, shutil, subprocess

# ── Re-define IDF builder here so this cell is self-contained ─────────────────
def _sf(name, wd, we):
    rows = ["Schedule:Compact,\n  {},\n  Fraction,\n  Through: 12/31,".format(name),
            "  For: Weekdays,"]
    for h, v in enumerate(wd, 1):
        rows.append("  Until: {:02d}:00,{},".format(h, v))
    rows.append("  For: AllOtherDays,")
    for h, v in enumerate(we, 1):
        rows.append("  Until: {:02d}:00,{}{}".format(h, v, "," if h < 24 else ";"))
    return "\n".join(rows)

def _st(name, wd, we):
    rows = ["Schedule:Compact,\n  {},\n  Temperature,\n  Through: 12/31,".format(name),
            "  For: Weekdays,"]
    for h, v in enumerate(wd, 1):
        rows.append("  Until: {:02d}:00,{},".format(h, v))
    rows.append("  For: AllOtherDays,")
    for h, v in enumerate(we, 1):
        rows.append("  Until: {:02d}:00,{}{}".format(h, v, "," if h < 24 else ";"))
    return "\n".join(rows)

def _make_idf(heating_sp=18.0, cooling_sp=26.0, heating_sb=15.0, cooling_sb=28.0):
    bui = build_bui()
    bp  = bui["building_parameters"]
    ig  = {g["name"]: g for g in bp["internal_gains"]}
    occ_wd = ig["occupants"]["weekday"];  occ_we = ig["occupants"]["weekend"]
    app_wd = ig["appliances"]["weekday"]; app_we = ig["appliances"]["weekend"]
    lit_wd = ig["lighting"]["weekday"];   lit_we = ig["lighting"]["weekend"]
    h_wd = bp["heating_profile"]["weekday"]; h_we = bp["heating_profile"]["weekend"]
    c_wd = bp["cooling_profile"]["weekday"]; c_we = bp["cooling_profile"]["weekend"]
    hsp_wd = [heating_sp if v > 0 else heating_sb for v in h_wd]
    hsp_we = [heating_sp if v > 0 else heating_sb for v in h_we]
    csp_wd = [cooling_sp if v > 0 else cooling_sb for v in c_wd]
    csp_we = [cooling_sp if v > 0 else cooling_sb for v in c_we]
    R_ext = round(max(0.001, 1/1.00 - 0.13 - 0.04), 4)
    R_int = round(max(0.001, 1/2.50 - 0.13 - 0.13), 4)
    R_slb = round(max(0.001, 1/1.80 - 0.13 - 0.13), 4)
    LN = 5.0; LE = 4.0; H = 2.7
    scheds = "\n\n".join([
        _sf("Occ_Sched", occ_wd, occ_we), _sf("App_Sched", app_wd, app_we),
        _sf("Lit_Sched", lit_wd, lit_we), _st("Heat_SP",   hsp_wd, hsp_we),
        _st("Cool_SP",   csp_wd, csp_we),
        "Schedule:Compact,\n  Always1,\n  Fraction,\n  Through: 12/31,\n  For: AllDays,\n  Until: 24:00,1.0;\n\n""Schedule:Compact,\n  Always4,\n  Any Number,\n  Through: 12/31,\n  For: AllDays,\n  Until: 24:00,4.0;",
        "Schedule:Compact,\n  Activity_W,\n  Any Number,\n  Through: 12/31,\n  For: AllDays,\n  Until: 24:00,160.0;",
    ])
    p = []
    p.append(
        "  Version, 24.1;\n\n  SimulationControl, Yes, No, No, No, Yes;\n\n"
        "  Building,\n    Apt_305_50_Barry_St_Carlton,\n    0.0, Suburbs, 0.04, 0.004,\n"
        "    FullInteriorAndExterior, 25, 6;\n\n  Timestep, 6;\n\n"
        "  GlobalGeometryRules,\n    UpperLeftCorner, CounterClockWise, World;\n\n"
        "  Site:Location,\n    Melbourne_VIC_AUS,\n    -37.800, 144.968, 10.0, 31.0;\n\n"
        "  RunPeriod,\n    FullYear,\n    1, 1, , 12, 31, ,\n    Sunday, Yes, Yes, No, Yes, Yes;\n\n"
        "  ScheduleTypeLimits, Fraction, 0.0, 1.0, Continuous, Dimensionless;\n"
        "  ScheduleTypeLimits, Temperature, -100, 200, Continuous, Temperature;\n"
        "  ScheduleTypeLimits, Any Number, -1e10, 1e10, Continuous;"
    )
    p.append(scheds)
    p.append("  Zone, Apt305, 0.0, 0.0, 0.0, 0.0, 1, 1, {H}, {V}, {A};".format(H=H, V=LE*LN*H, A=LE*LN))
    p.append(
        "  Material:NoMass, Mat_ExtWall, MediumRough,  {re}, 0.9, 0.75, 0.75;\n"
        "  Material:NoMass, Mat_IntWall, MediumSmooth, {ri}, 0.9, 0.0,  0.0;\n"
        "  Material:NoMass, Mat_IntSlab, MediumSmooth, {rs}, 0.9, 0.0,  0.0;\n"
        "  WindowMaterial:SimpleGlazingSystem, Mat_Window, 5.40, 0.65;\n\n"
        "  Construction, Con_ExtWall, Mat_ExtWall;\n  Construction, Con_IntWall, Mat_IntWall;\n"
        "  Construction, Con_IntSlab, Mat_IntSlab;\n  Construction, Con_Window,  Mat_Window;".format(re=R_ext, ri=R_int, rs=R_slb)
    )
    p.append("  SurfaceProperty:OtherSideCoefficients,\n    OSC_Adj21C,\n    8.0,\n    21.0,\n    1.0, 0.0, 0.0, 0.0, 0.0;")
    p.append(
        "  BuildingSurface:Detailed,\n    WestWall, Wall, Con_ExtWall, Apt305, ,\n"
        "    Outdoors, , SunExposed, WindExposed, , 4,\n"
        "    0.0,{LN},{H},\n    0.0,{LN},0.0,\n    0.0,0.0,0.0,\n    0.0,0.0,{H};\n\n"
        "  BuildingSurface:Detailed,\n    NorthWall, Wall, Con_IntWall, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj21C, NoSun, NoWind, , 4,\n"
        "    {LE},{LN},{H},\n    {LE},{LN},0.0,\n    0.0,{LN},0.0,\n    0.0,{LN},{H};\n\n"
        "  BuildingSurface:Detailed,\n    EastWall, Wall, Con_IntWall, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj21C, NoSun, NoWind, , 4,\n"
        "    {LE},0.0,{H},\n    {LE},0.0,0.0,\n    {LE},{LN},0.0,\n    {LE},{LN},{H};\n\n"
        "  BuildingSurface:Detailed,\n    SouthWall, Wall, Con_IntWall, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj21C, NoSun, NoWind, , 4,\n"
        "    0.0,0.0,{H},\n    0.0,0.0,0.0,\n    {LE},0.0,0.0,\n    {LE},0.0,{H};\n\n"
        "  BuildingSurface:Detailed,\n    FloorSlab, Floor, Con_IntSlab, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj21C, NoSun, NoWind, , 4,\n"
        "    0.0,0.0,0.0,\n    0.0,{LN},0.0,\n    {LE},{LN},0.0,\n    {LE},0.0,0.0;\n\n"
        "  BuildingSurface:Detailed,\n    CeilingSlab, Ceiling, Con_IntSlab, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj21C, NoSun, NoWind, , 4,\n"
        "    0.0,0.0,{H},\n    {LE},0.0,{H},\n    {LE},{LN},{H},\n    0.0,{LN},{H};\n\n"
        "  FenestrationSurface:Detailed,\n    WestWin_Fixed, Window, Con_Window, WestWall, , , , 1.0, 4,\n"
        "    0.0,{y1h},1.9,  0.0,{y1h},1.0,  0.0,{y1l},1.0,  0.0,{y1l},1.9;\n\n"
        "  FenestrationSurface:Detailed,\n    WestWin_Operable, Window, Con_Window, WestWall, , , , 1.0, 4,\n"
        "    0.0,{y2h},1.9,  0.0,{y2h},1.0,  0.0,{y2l},1.0,  0.0,{y2l},1.9;".format(
            H=H, LN=LN, LE=LE, y1h=LN-1.0, y1l=LN-1.9, y2h=LN-2.5, y2l=LN-3.4)
    )
    # People: Method, LightingLevel(blank), PeoplePerArea, ZoneAreaPerPerson(blank), FracRadiant, SensHeatFrac(blank), ActivitySched
    # Lights: Method, LightingLevel(blank), W/Area, W/Person(blank), ReturnAirFrac, FracRadiant, FracVisible
    #         ReturnAirFrac=0.0, FracRadiant=0.72, FracVisible=0.18  =>  sum=0.90 < 1.0  OK
    p.append(
        "  People,\n    Apt305_People, Apt305, Occ_Sched,\n"
        "    People/Area, , 0.1, , 0.3, , Activity_W;\n\n"
        # 0.1 people/m2 x 20 m2 = 2 people
        "  ElectricEquipment,\n    Apt305_Appliances, Apt305, App_Sched,\n"
        "    Watts/Area, , 5.0, , 0.0, 0.5, 0.0;\n\n"
        "  Lights,\n    Apt305_Lights, Apt305, Lit_Sched,\n"
        "    Watts/Area, , 3.0, , 0.0, 0.72, 0.18;"
    )
    p.append("  DesignSpecification:OutdoorAir,\n    Apt305_OA, Flow/Area, 0.002, , , , Always1;")
    p.append(
        "  ZoneHVAC:IdealLoadsAirSystem,\n    Apt305_IdealLoads, ,\n"
        "    Apt305_SupplyAir, Apt305_ExhaustAir, ,\n"
        "    50, 12, 0.010, 0.010,\n    NoLimit, , , NoLimit, , ,\n    , ,\n"
        "    ConstantSupplyHumidityRatio, ,\n    ConstantSupplyHumidityRatio,\n"
        "    Apt305_OA, , , , , , ;\n\n"
        "  ZoneHVAC:EquipmentList,\n    Apt305_EqpList,\n    SequentialLoad,\n"
        "    ZoneHVAC:IdealLoadsAirSystem, Apt305_IdealLoads, 1, 1;\n\n"
        "  ZoneHVAC:EquipmentConnections,\n    Apt305, Apt305_EqpList,\n"
        "    Apt305_SupplyAir, Apt305_ExhaustAir,\n    Apt305_ZoneAirNode;\n\n"
        "  NodeList, Apt305_SupplyAir, Apt305_SupplyAir_Node;"
    )
    p.append(
        "  ThermostatSetpoint:DualSetpoint, Apt305_DualSP, Heat_SP, Cool_SP;\n\n"
        "  ZoneControl:Thermostat,\n    Apt305_Thermostat, Apt305, Always4,\n"
        "    ThermostatSetpoint:DualSetpoint, Apt305_DualSP;"
    )
    p.append(
        "  OutputControl:Table:Style, HTML;\n  Output:Table:SummaryReports, AllSummary;\n"
        "  Output:VariableDictionary, IDF;\n"
        "  Output:Variable, *, Zone Mean Air Temperature, Hourly;\n"
        "  Output:Variable, *, Zone Ideal Loads Supply Air Total Heating Energy, Hourly;\n"
        "  Output:Variable, *, Zone Ideal Loads Supply Air Total Cooling Energy, Hourly;\n"
        "  Output:Diagnostics, DisplayAllWarnings;"
    )
    return "\n\n".join(p)

# ── Write the IDF ─────────────────────────────────────────────────────────────
IDF_PATH = "/content/apt305.idf"
idf_text = _make_idf()
with open(IDF_PATH, "w") as _f:
    _f.write(idf_text)
assert "GlobalGeometryRules" in idf_text
assert "AllOtherDays" in idf_text
assert "0.0, 0.72, 0.18" in idf_text        # Lights fractions sum < 1
assert "People/Area, , 0.1" in idf_text     # 2 people / 20 m² = 0.1 p/m²
assert "Always4" in idf_text                 # DualSetpoint needs ctrl-type=4
assert "Supply Air Total" in idf_text        # correct output variable names
print(f"IDF written  ({idf_text.count(chr(10))} lines)  — all assertions OK")

# ── Run EnergyPlus ────────────────────────────────────────────────────────────
EP_OUT_DIR = "/content/ep_output"
EP_BIN     = os.path.join(EP_DIR, "energyplus")
if os.path.exists(EP_OUT_DIR):
    shutil.rmtree(EP_OUT_DIR)
os.makedirs(EP_OUT_DIR)

print("\n=== EnergyPlus simulation ===")
result = subprocess.run(
    [EP_BIN, "-w", EPW_PATH, "-d", EP_OUT_DIR, "-p", "apt305", IDF_PATH],
    capture_output=True, text=True,
)
print("── STDOUT ─────────────────────────────────────────────────")
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
print("── STDERR ─────────────────────────────────────────────────")
print(result.stderr[-1000:] if len(result.stderr) > 1000 else result.stderr)

print(f"\n{'='*60}")
err_files = glob.glob(f"{EP_OUT_DIR}/*.err")
if err_files:
    with open(err_files[0]) as f:
        err_lines = f.readlines()
    print("EnergyPlus .err log — last 120 lines:")
    print("".join(err_lines[-120:]))
else:
    print(f"No .err file. Files: {os.listdir(EP_OUT_DIR)}")
print("="*60)

if result.returncode != 0:
    raise RuntimeError(f"EnergyPlus exited with code {result.returncode}.")

print("\nSimulation complete  ✓")
print("All output files:", os.listdir(EP_OUT_DIR))

In [ ]:
# ── Parse .eso to extract annual heating / cooling energy ────────────────────
import re

ESO_PATH = "/content/ep_output/apt305out.eso"

def parse_eso_annual(eso_path,
                     heat_var="Zone Ideal Loads Supply Air Total Heating Energy",
                     cool_var="Zone Ideal Loads Supply Air Total Cooling Energy"):
    """Sum all hourly records for the given variables from an EnergyPlus .eso file."""
    var_ids  = {}
    heat_j   = 0.0
    cool_j   = 0.0
    in_dict  = True

    with open(eso_path) as f:
        for line in f:
            line = line.rstrip()
            if in_dict:
                if line.strip() == "End of Data Dictionary":
                    in_dict = False
                    continue
                m = re.match(r'''^\ *(\d+),\d+,.*?,(.+?)\s*\[''', line)
                if m:
                    vname = m.group(2).strip()
                    if heat_var.lower() in vname.lower():
                        var_ids[m.group(1)] = "heat"
                    elif cool_var.lower() in vname.lower():
                        var_ids[m.group(1)] = "cool"
            else:
                parts = line.split(",", 1)
                if len(parts) == 2 and parts[0].strip() in var_ids:
                    try:
                        v = float(parts[1])
                        if var_ids[parts[0].strip()] == "heat":
                            heat_j += v
                        else:
                            cool_j += v
                    except ValueError:
                        pass

    return heat_j, cool_j

heat_j, cool_j = parse_eso_annual(ESO_PATH)

EP_AREA_m2     = 5.0 * 4.0
EP_heat_kWh    = heat_j / 3_600_000.0
EP_cool_kWh    = cool_j / 3_600_000.0
EP_heat_kWh_m2 = EP_heat_kWh / EP_AREA_m2
EP_cool_kWh_m2 = EP_cool_kWh / EP_AREA_m2

print(f"EnergyPlus 24.1  |  Heating: {EP_heat_kWh:.1f} kWh  ({EP_heat_kWh_m2:.1f} kWh/m²)")
print(f"EnergyPlus 24.1  |  Cooling: {EP_cool_kWh:.1f} kWh  ({EP_cool_kWh_m2:.1f} kWh/m²)")

# ── Extend comparison table ───────────────────────────────────────────────────
comparison_df_3 = comparison_df._append({
    "engine":        "EnergyPlus 24.1.0",
    "setpoints":     "18°C / 26°C  (DualSetpoint)",
    "heating_kWh":   round(EP_heat_kWh, 1),
    "cooling_kWh":   round(EP_cool_kWh, 1),
    "heating_kWh_m2": round(EP_heat_kWh_m2, 1),
    "cooling_kWh_m2": round(EP_cool_kWh_m2, 1),
}, ignore_index=True)

print("\n=== Three-engine comparison ===")
print(comparison_df_3[["engine","heating_kWh","cooling_kWh",
                         "heating_kWh_m2","cooling_kWh_m2"]].to_string(index=False))


In [ ]:
# ── Updated bar chart including EnergyPlus ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(comparison_df_3))
width = 0.35

bars_h = ax.bar(x - width / 2, comparison_df_3["heating_kWh"], width, label="Heating need")
bars_c = ax.bar(x + width / 2, comparison_df_3["cooling_kWh"], width, label="Cooling need")

for bar in bars_h:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, h + 15, f"{h:.0f}", ha="center", va="bottom", fontsize=8)
for bar in bars_c:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, h + 15, f"{h:.0f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(comparison_df_3["engine"], rotation=12, ha="right", fontsize=9)
ax.set_ylabel("Annual energy need (kWh/yr)")
ax.set_title("Apt 305, 50 Barry St, Carlton\nAll three engines compared")
ax.legend()
plt.tight_layout()
plt.show()

# ── Save extended CSV ─────────────────────────────────────────────────────────
OUTPUT_PATH_3 = "/content/engine_comparison_apt305_all3.csv"
comparison_df_3.to_csv(OUTPUT_PATH_3, index=False)
print(f"Saved: {OUTPUT_PATH_3}")
print()
print(comparison_df_3.to_string(index=False))

In [ ]:
# ── V1: Determinism ───────────────────────────────────────────────────────────
import numpy as np

PASS = "✅ PASS"
FAIL = "❌ FAIL"

print("=" * 62)
print("V1  DETERMINISM — same input must give bit-exact output")
print("=" * 62)

# ── ISO engines: run twice and compare Q_HC series ───────────────────────────
def _run_orig():
    bui, _ = orig_sanitize(build_bui(), fix=True)
    out = OrigISO52016.Temperature_and_Energy_needs_calculation(
        bui, weather_source=WEATHER_SOURCE)
    return (out[0] if len(out) == 3 else out[0])["Q_HC"].values

def _run_mod():
    bui_m = build_bui()
    sp = ncc_setpoints
    bui_m["building_parameters"]["temperature_setpoints"].update({
        "heating_setpoint": sp["heating_setpoint"],
        "heating_setback": sp["heating_setback"],
        "cooling_setpoint": sp["cooling_setpoint_living"],
        "cooling_setback": sp["cooling_setback"],
    })
    bui_m, _ = mod_sanitize(bui_m, fix=True)
    out = ModISO52016.Temperature_and_Energy_needs_calculation(
        bui_m, weather_source=WEATHER_SOURCE)
    return (out[0] if len(out) == 3 else out[0])["Q_HC"].values

print("\nRunning Original engine  x2 ...", end=" ", flush=True)
r1a, r1b = _run_orig(), _run_orig()
match1 = np.array_equal(r1a, r1b)
print(PASS if match1 else FAIL)
if not match1:
    diff_idx = np.where(r1a != r1b)[0]
    print(f"  Differing timesteps: {len(diff_idx)}  (first at index {diff_idx[0]})")

print("Running Modified engine  x2 ...", end=" ", flush=True)
r2a, r2b = _run_mod(), _run_mod()
match2 = np.array_equal(r2a, r2b)
print(PASS if match2 else FAIL)
if not match2:
    diff_idx = np.where(r2a != r2b)[0]
    print(f"  Differing timesteps: {len(diff_idx)}  (first at index {diff_idx[0]})")

# ── EnergyPlus: re-run and compare parsed annual totals ──────────────────────
import glob, os, re, shutil, subprocess

def _ep_run_and_parse(tag="run"):
    out_dir = f"/content/ep_val_{tag}"
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    r = subprocess.run(
        [os.path.join(EP_DIR, "energyplus"), "-w", EPW_PATH,
         "-d", out_dir, "-p", "apt305", IDF_PATH],
        capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"EnergyPlus failed (tag={tag})")
    h_j, c_j = _parse_eso(os.path.join(out_dir, "apt305out.eso"))
    shutil.rmtree(out_dir)
    return h_j, c_j

def _parse_eso(path,
               hv="Zone Ideal Loads Supply Air Total Heating Energy",
               cv="Zone Ideal Loads Supply Air Total Cooling Energy"):
    var_ids, h, c, in_dict = {}, 0.0, 0.0, True
    with open(path) as f:
        for line in f:
            line = line.rstrip()
            if in_dict:
                if line.strip() == "End of Data Dictionary":
                    in_dict = False; continue
                m = re.match(r'^ *(\d+),\d+,.*?,(.+?)\s*\[', line)
                if m:
                    n = m.group(2).strip()
                    if hv.lower() in n.lower(): var_ids[m.group(1)] = "h"
                    elif cv.lower() in n.lower(): var_ids[m.group(1)] = "c"
            else:
                p = line.split(",", 1)
                if len(p) == 2 and p[0].strip() in var_ids:
                    try:
                        v = float(p[1])
                        if var_ids[p[0].strip()] == "h": h += v
                        else: c += v
                    except ValueError: pass
    return h, c

print("Running EnergyPlus       x2 ...", end=" ", flush=True)
ep_h1, ep_c1 = _ep_run_and_parse("v1a")
ep_h2, ep_c2 = _ep_run_and_parse("v1b")
ep_match = (ep_h1 == ep_h2) and (ep_c1 == ep_c2)
print(PASS if ep_match else FAIL)
if not ep_match:
    print(f"  Heat diff: {abs(ep_h1-ep_h2)/3.6e6:.4f} kWh  |  Cool diff: {abs(ep_c1-ep_c2)/3.6e6:.4f} kWh")

print("\n── Summary ──────────────────────────────────────────────")
for name, ok in [("Original pyBuildingEnergy", match1),
                 ("Modified AIB fork",         match2),
                 ("EnergyPlus 24.1.0",         ep_match)]:
    print(f"  {PASS if ok else FAIL}  {name}")



In [ ]:
import pandas as pd

print("=" * 62)
print("V2  ANNUAL ENERGY BALANCE")
print("=" * 62)

def _balance_report(engine_name, q_hc_kwh_net, q_tr=None, q_ve=None,
                    q_sol=None, q_int=None):
    """Print balance for one engine.  All values in kWh, positive = gain to zone."""
    if None in (q_tr, q_ve, q_sol, q_int):
        print(f"\n{engine_name}: insufficient component data — skipping balance check")
        return
    lhs = q_hc_kwh_net
    rhs = q_tr + q_ve - q_sol - q_int
    throughput = abs(q_tr) + abs(q_ve) + abs(q_sol) + abs(q_int)
    residual_pct = 100.0 * abs(lhs - rhs) / max(throughput, 1.0)
    tag = "✅" if residual_pct < 2.0 else ("⚠️" if residual_pct < 10.0 else "❌")
    print(f"\n{engine_name}:")
    print(f"  Q_HC (net, LHS)    = {lhs:+10.1f} kWh")
    print(f"  Q_tr (transmission)= {q_tr:+10.1f} kWh")
    print(f"  Q_ve (ventilation) = {q_ve:+10.1f} kWh")
    print(f"  Q_sol (solar gains)= {q_sol:+10.1f} kWh")
    print(f"  Q_int (int. gains) = {q_int:+10.1f} kWh")
    print(f"  RHS                = {rhs:+10.1f} kWh")
    print(f"  Residual           = {residual_pct:.2f}%  {tag}")

# ── Helper: extract flows from Sankey dict ───────────────────────────────────
def _from_sankey(sankey, q_hc_net, engine_name):
    if not sankey:
        return None, None, None, None
    print(f"\n  [{engine_name}] Sankey keys: {list(sankey.keys())}")
    # Common key patterns used by pyBuildingEnergy Sankey output
    def _get(*keys):
        for k in keys:
            if k in sankey: return sankey[k] / 1000.0  # W·h → kWh
        return None
    q_tr  = _get("Q_tr", "transmission", "Q_transmission")
    q_ve  = _get("Q_ve", "ventilation",  "Q_ventilation")
    q_sol = _get("Q_sol","solar",        "Q_solar")
    q_int = _get("Q_int","internal",     "Q_internal")
    return q_tr, q_ve, q_sol, q_int

# ── Helper: inspect hourly DataFrame columns ─────────────────────────────────
def _from_hourly(hourly_df, q_hc_net, engine_name):
    cols = [c for c in hourly_df.columns if c != "Q_HC"]
    print(f"\n  [{engine_name}] Hourly DataFrame columns: {cols}")
    def _sumcol(*keys):
        for k in keys:
            if k in hourly_df.columns: return hourly_df[k].sum() / 1000.0
        return None
    q_tr  = _sumcol("Q_tr", "Q_transmission", "H_tr")
    q_ve  = _sumcol("Q_ve", "Q_ventilation",  "H_ve")
    q_sol = _sumcol("Q_sol","Q_solar",        "phi_sol")
    q_int = _sumcol("Q_int","Q_internal",     "phi_int")
    return q_tr, q_ve, q_sol, q_int

# ── Original engine ───────────────────────────────────────────────────────────
q_hc_orig_net = (Q_HC_original[Q_HC_original > 0].sum()
                 - Q_HC_original[Q_HC_original < 0].abs().sum()) / 1000.0
if sankey_original:
    args = _from_sankey(sankey_original, q_hc_orig_net, "Original")
else:
    args = _from_hourly(hourly_original, q_hc_orig_net, "Original")
_balance_report("Original pyBuildingEnergy (PyPI)", q_hc_orig_net, *args)

# ── Modified engine ───────────────────────────────────────────────────────────
q_hc_mod_net = (Q_HC_modified[Q_HC_modified > 0].sum()
                - Q_HC_modified[Q_HC_modified < 0].abs().sum()) / 1000.0
if sankey_modified:
    args = _from_sankey(sankey_modified, q_hc_mod_net, "Modified")
else:
    args = _from_hourly(hourly_modified, q_hc_mod_net, "Modified")
_balance_report("Modified AIB fork", q_hc_mod_net, *args)

# ── EnergyPlus — parse HTML AllSummary ───────────────────────────────────────
HTM_PATH = "/content/ep_output/apt305tbl.htm"
print(f"\nEnergyPlus 24.1.0 (HTML summary from {HTM_PATH}):")
try:
    tables = pd.read_html(HTM_PATH)
    # Find the Zone Component Loads or Site:EnergyUse table
    ep_q_sol = ep_q_int = ep_q_tr = ep_q_ve = None
    for tbl in tables:
        tbl.columns = [str(c).strip() for c in tbl.columns]
        first_col = tbl.iloc[:, 0].astype(str).str.strip()
        # Zone Component Loads Summary table contains these row labels
        for i, label in enumerate(first_col):
            lab = label.lower()
            # Solar gains
            if "window" in lab and ("heat gain" in lab or "solar" in lab):
                try: ep_q_sol = float(str(tbl.iloc[i, 1]).replace(",", "")) / 3600.0
                except: pass
            # People + Equipment + Lights → internal gains
            if "people" in lab and "total" in lab:
                try: ep_q_int = (ep_q_int or 0) + float(str(tbl.iloc[i, 1]).replace(",",""))/3600
                except: pass
            if "lighting" in lab and "total" in lab:
                try: ep_q_int = (ep_q_int or 0) + float(str(tbl.iloc[i, 1]).replace(",",""))/3600
                except: pass
            if "equipment" in lab and "total" in lab:
                try: ep_q_int = (ep_q_int or 0) + float(str(tbl.iloc[i, 1]).replace(",",""))/3600
                except: pass
    if None in (ep_q_sol, ep_q_int):
        print("  Component table not found in HTML — printing table titles for inspection:")
        for t in tables[:20]:
            print("   ", list(t.iloc[:, 0])[:4])
    else:
        ep_q_hc_net = (EP_heat_kWh - EP_cool_kWh)
        _balance_report("EnergyPlus 24.1.0", ep_q_hc_net,
                        q_tr=None, q_ve=None, q_sol=ep_q_sol, q_int=ep_q_int)
        print("  (Transmission/ventilation split not yet extracted from HTML — balance shown with available flows)")
except Exception as e:
    print(f"  Could not parse HTML report: {e}")

In [ ]:
# ── V3 + V4: Limit behaviour ─────────────────────────────────────────────────
# V3: near-perfect insulation (U=0.001 W/m²K) + zero internal gains
#     → heating + cooling should drop > 90 % vs baseline
#     (residual driven only by ventilation losses to outdoor air)
# V4: normal insulation + zero internal gains
#     → cooling demand drops (no heat sources), heating demand rises slightly

import copy, os, re, shutil, subprocess, numpy as np

PASS = "✅"; WARN = "⚠️"; FAIL = "❌"
print("=" * 62)
print("V3 + V4  LIMIT BEHAVIOUR")
print("=" * 62)

# ── Baseline kWh (from main cells) ───────────────────────────────────────────
base = {
    "orig":  (heating_kWh_original, cooling_kWh_original),
    "mod":   (heating_kWh_modified, cooling_kWh_modified),
    "ep":    (EP_heat_kWh,          EP_cool_kWh),
}

# ── Helper: build modified bui for ISO engines ────────────────────────────────
def _mod_bui(bui, zero_u=False, zero_gains=False):
    b = copy.deepcopy(bui)
    if zero_u:
        for surf in b["building_surface"]:
            surf["u_value"] = 0.001
        for az in b.get("adjacent_zones", []):
            az["transmittance_U_elements"] = az["transmittance_U_elements"] * 0.0 + 0.001
    if zero_gains:
        for g in b["building_parameters"]["internal_gains"]:
            g["full_load"] = 0.0
    return b

def _iso_demand(bui_dict, engine_cls, sanitize_fn):
    b, _ = sanitize_fn(bui_dict, fix=True)
    out = engine_cls.Temperature_and_Energy_needs_calculation(
        b, weather_source=WEATHER_SOURCE)
    qhc = (out[0] if len(out) == 3 else out[0])["Q_HC"]
    return qhc[qhc > 0].sum()/1000, -qhc[qhc < 0].sum()/1000

# ── Helper: build modified IDF for EnergyPlus ────────────────────────────────
def _ep_limit_demand(zero_u=False, zero_gains=False, tag="lim"):
    idf = _make_idf()
    if zero_u:
        # Replace R-values with 999 (≈ U=0.001 W/m²K)
        idf = re.sub(r'(Mat_ExtWall, MediumRough,\s+)[\d.]+',  r'\g<1>999.0', idf)
        idf = re.sub(r'(Mat_IntWall, MediumSmooth,\s+)[\d.]+', r'\g<1>999.0', idf)
        idf = re.sub(r'(Mat_IntSlab, MediumSmooth,\s+)[\d.]+', r'\g<1>999.0', idf)
        # Also set window U to near-zero via override
        idf = idf.replace("WindowMaterial:SimpleGlazingSystem, Mat_Window, 5.40, 0.65;",
                           "WindowMaterial:SimpleGlazingSystem, Mat_Window, 0.001, 0.65;")
    if zero_gains:
        idf = re.sub(r'(Watts/Area, , )[\d.]+,', r'\g<1>0.0,', idf)      # Equipment & Lights
        idf = re.sub(r'(People/Area, , )[\d.]+,', r'\g<1>0.0,', idf)     # People
    tmp_idf = f"/content/apt305_{tag}.idf"
    out_dir = f"/content/ep_{tag}"
    with open(tmp_idf, "w") as f: f.write(idf)
    if os.path.exists(out_dir): shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    r = subprocess.run(
        [os.path.join(EP_DIR, "energyplus"), "-w", EPW_PATH,
         "-d", out_dir, "-p", "apt305", tmp_idf],
        capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"EnergyPlus failed for tag={tag}:\n{r.stderr[-500:]}")
    h, c = _parse_eso(os.path.join(out_dir, "apt305out.eso"))
    shutil.rmtree(out_dir)
    os.remove(tmp_idf)
    return h/3.6e6, c/3.6e6

def _check(label, h_new, c_new, h_base, c_base, drop_threshold=0.90):
    total_base = h_base + c_base
    total_new  = h_new  + c_new
    if total_base < 1.0:
        print(f"  {label}: baseline near-zero, skip relative check")
        return
    drop = 1.0 - total_new / total_base
    icon = PASS if drop >= drop_threshold else (WARN if drop >= 0.5 else FAIL)
    print(f"  {icon} {label:40s}  demand dropped {drop*100:.1f}%"
          f"  (H={h_new:.1f} kWh  C={c_new:.1f} kWh)")

# ── V3: U → 0, gains → 0 ──────────────────────────────────────────────────────
print("\n── V3: near-perfect insulation + zero internal gains ──────")
print("   (expect > 90% drop in total heating + cooling demand)\n")

bui_v3_orig = _mod_bui(build_bui(), zero_u=True, zero_gains=True)
h3o, c3o = _iso_demand(bui_v3_orig, OrigISO52016, orig_sanitize)
_check("Original pyBuildingEnergy", h3o, c3o, *base["orig"])

bui_v3_mod = build_bui()
sp = ncc_setpoints
bui_v3_mod["building_parameters"]["temperature_setpoints"].update({
    "heating_setpoint": sp["heating_setpoint"], "heating_setback": sp["heating_setback"],
    "cooling_setpoint": sp["cooling_setpoint_living"], "cooling_setback": sp["cooling_setback"],
})
bui_v3_mod = _mod_bui(bui_v3_mod, zero_u=True, zero_gains=True)
h3m, c3m = _iso_demand(bui_v3_mod, ModISO52016, mod_sanitize)
_check("Modified AIB fork", h3m, c3m, *base["mod"])

print("  Running EnergyPlus limit case (U→0, gains=0) ...", end=" ", flush=True)
h3e, c3e = _ep_limit_demand(zero_u=True, zero_gains=True, tag="v3")
_check("EnergyPlus 24.1.0", h3e, c3e, *base["ep"])

# ── V4: normal U, gains → 0 ───────────────────────────────────────────────────
print("\n── V4: zero internal gains (normal insulation) ────────────")
print("   (expect cooling to drop; heating may rise slightly)\n")

bui_v4_orig = _mod_bui(build_bui(), zero_u=False, zero_gains=True)
h4o, c4o = _iso_demand(bui_v4_orig, OrigISO52016, orig_sanitize)
print(f"  Original pyBuildingEnergy  →  H={h4o:.1f} kWh  C={c4o:.1f} kWh"
      f"  (base H={base['orig'][0]:.1f}  C={base['orig'][1]:.1f})")

bui_v4_mod = build_bui()
bui_v4_mod["building_parameters"]["temperature_setpoints"].update({
    "heating_setpoint": sp["heating_setpoint"], "heating_setback": sp["heating_setback"],
    "cooling_setpoint": sp["cooling_setpoint_living"], "cooling_setback": sp["cooling_setback"],
})
bui_v4_mod = _mod_bui(bui_v4_mod, zero_u=False, zero_gains=True)
h4m, c4m = _iso_demand(bui_v4_mod, ModISO52016, mod_sanitize)
print(f"  Modified AIB fork          →  H={h4m:.1f} kWh  C={c4m:.1f} kWh"
      f"  (base H={base['mod'][0]:.1f}  C={base['mod'][1]:.1f})")

print("  Running EnergyPlus (gains=0) ...", end=" ", flush=True)
h4e, c4e = _ep_limit_demand(zero_u=False, zero_gains=True, tag="v4")
print(f"\n  EnergyPlus 24.1.0          →  H={h4e:.1f} kWh  C={c4e:.1f} kWh"
      f"  (base H={base['ep'][0]:.1f}  C={base['ep'][1]:.1f})")

# ── Conservation check ────────────────────────────────────────────────────────
print("\n── Conservation check ──────────────────────────────────────")
print("  With U→0 and gains→0, only ventilation to outdoor air remains.")
print("  Remaining demand should be ≈ ACH × V × ρCp × ΔT.")
ACH  = 0.5          # rough infiltration air changes/hr (Melbourne avg ΔT≈10°C)
V    = 5.0*4.0*2.7  # zone volume m³
rho_cp = 0.33       # Wh/m³K
est_vent_kwh = ACH * V * rho_cp * 10.0 * 8760 / 1000.0
print(f"  Estimated ventilation loss ≈ {est_vent_kwh:.0f} kWh/yr")
print(f"  EP V3 total demand         = {h3e+c3e:.1f} kWh/yr")


In [ ]:
# ── V5: Seasonal pattern check ────────────────────────────────────────────────
# Melbourne (Southern Hemisphere):
#   ✅ Heating peaks: June–August  (winter)
#   ✅ Cooling peaks: December–February  (summer)
# Test for all three engines using their hourly data.

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np, re, pandas as pd

print("=" * 62)
print("V5  SEASONAL PATTERN — Melbourne heating/cooling by month")
print("=" * 62)

MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

# ── ISO engines: hourly Q_HC → monthly ───────────────────────────────────────
def _iso_monthly(qhc_series):
    """Convert 8760-length annual Q_HC series to (12,) heating and cooling arrays."""
    hours_per_month = [744,672,744,720,744,720,744,744,720,744,720,744]  # non-leap 2017
    h_kwh = np.zeros(12); c_kwh = np.zeros(12)
    idx = 0
    for m, h in enumerate(hours_per_month):
        slc = qhc_series.iloc[idx:idx+h]
        h_kwh[m] = slc[slc > 0].sum() / 1000.0
        c_kwh[m] = -slc[slc < 0].sum() / 1000.0
        idx += h
    return h_kwh, c_kwh

orig_h_m, orig_c_m = _iso_monthly(Q_HC_original)
mod_h_m,  mod_c_m  = _iso_monthly(Q_HC_modified)

# ── EnergyPlus: parse .eso with date tracking ─────────────────────────────────
def _ep_monthly(eso_path,
                hv="Zone Ideal Loads Supply Air Total Heating Energy",
                cv="Zone Ideal Loads Supply Air Total Cooling Energy"):
    """Parse .eso and return monthly heating/cooling arrays (kWh)."""
    var_ids = {}; h_m = np.zeros(12); c_m = np.zeros(12)
    in_dict = True; current_month = 0

    with open(eso_path) as f:
        for line in f:
            line = line.rstrip()
            if in_dict:
                if line.strip() == "End of Data Dictionary":
                    in_dict = False; continue
                m = re.match(r'^ *(\d+),\d+,.*?,(.+?)\s*\[', line)
                if m:
                    n = m.group(2).strip()
                    if hv.lower() in n.lower(): var_ids[m.group(1)] = "h"
                    elif cv.lower() in n.lower(): var_ids[m.group(1)] = "c"
            else:
                # Hourly time-stamp record:  2,N,MM/DD,HH:MM:SS,...
                if line.startswith("2,"):
                    parts = line.split(",")
                    try:
                        date_str = parts[2].strip()        # "01/15"
                        current_month = int(date_str.split("/")[0]) - 1
                    except (IndexError, ValueError): pass
                    continue
                parts = line.split(",", 1)
                if len(parts) == 2 and parts[0].strip() in var_ids:
                    try:
                        v = float(parts[1]) / 3_600_000.0  # J → kWh
                        if var_ids[parts[0].strip()] == "h":
                            h_m[current_month] += v
                        else:
                            c_m[current_month] += v
                    except ValueError: pass
    return h_m, c_m

ESO = "/content/ep_output/apt305out.eso"
ep_h_m, ep_c_m = _ep_monthly(ESO)

# ── Seasonal peak check ───────────────────────────────────────────────────────
WINTER = {5, 6, 7}   # Jun Jul Aug (0-indexed)
SUMMER = {11, 0, 1}  # Dec Jan Feb

def _check_peaks(name, h_m, c_m):
    heat_peak = int(np.argmax(h_m))
    cool_peak = int(np.argmax(c_m))
    hok = heat_peak in WINTER
    cok = cool_peak in SUMMER
    h_icon = "✅" if hok else "❌"
    c_icon = "✅" if cok else "❌"
    print(f"  {name}:")
    print(f"    {h_icon} Heating peaks in {MONTHS[heat_peak]} ({h_m[heat_peak]:.0f} kWh)  — expect Jun/Jul/Aug")
    print(f"    {c_icon} Cooling peaks in {MONTHS[cool_peak]} ({c_m[cool_peak]:.0f} kWh)  — expect Dec/Jan/Feb")

print()
_check_peaks("Original pyBuildingEnergy", orig_h_m, orig_c_m)
_check_peaks("Modified AIB fork",         mod_h_m,  mod_c_m)
_check_peaks("EnergyPlus 24.1.0",         ep_h_m,   ep_c_m)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
datasets = [
    ("Original pyBuildingEnergy", orig_h_m, orig_c_m, "steelblue", "tomato"),
    ("Modified AIB fork",         mod_h_m,  mod_c_m,  "seagreen",  "darkorange"),
    ("EnergyPlus 24.1.0",         ep_h_m,   ep_c_m,   "purple",    "firebrick"),
]
x = np.arange(12)
for ax, (title, h_m, c_m, hcol, ccol) in zip(axes, datasets):
    ax.bar(x - 0.2, h_m, 0.38, label="Heating", color=hcol, alpha=0.85)
    ax.bar(x + 0.2, c_m, 0.38, label="Cooling", color=ccol, alpha=0.85)
    ax.set_title(title, fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(MONTHS, fontsize=7, rotation=45)
    ax.set_ylabel("kWh / month")
    ax.yaxis.set_major_locator(mticker.MaxNLocator(5))
    ax.legend(fontsize=7)
    # Shade winter (Jun–Aug) and summer (Dec–Feb) lightly
    for m in WINTER: ax.axvspan(m-0.5, m+0.5, alpha=0.07, color="blue")
    for m in [11]:   ax.axvspan(m-0.5, m+0.5, alpha=0.07, color="red")
    for m in [0, 1]: ax.axvspan(m-0.5, m+0.5, alpha=0.07, color="red")

fig.suptitle("V5 — Seasonal pattern: Apt 305, Melbourne\n"
             "(blue shading = winter, red shading = summer)", fontsize=10)
plt.tight_layout()
plt.show()
print("\nAll V5 checks complete.")

---
## 9. Monthly consumption chart — pyBuildingEnergy vs EnergyPlus

Interactive grouped bar chart (pyecharts) of monthly heating (`Q_H`) and cooling
(`Q_C`) for the ISO 52016 engine and EnergyPlus side by side, with value labels,
a toolbox (save / restore / data view / switch line-bar) and a zoom slider.

In [ ]:
# ── Monthly consumption: ISO 52016 vs EnergyPlus (pyecharts) ──────────────────
!pip install pyecharts -q

import numpy as np, re
from pyecharts.charts import Bar
from pyecharts import options as opts
from pyecharts.globals import ThemeType

MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
HOURS_PER_MONTH = [744,672,744,720,744,720,744,744,720,744,720,744]   # non-leap year

# ── ISO 52016 engine: hourly Q_HC (W) → monthly kWh ──────────────────────────
def iso_monthly(qhc_series):
    h = np.zeros(12); c = np.zeros(12); i = 0
    for m, n in enumerate(HOURS_PER_MONTH):
        s = qhc_series.iloc[i:i+n]
        h[m] = s[s > 0].sum() / 1000.0          # heating  (W·h → kWh)
        c[m] = -s[s < 0].sum() / 1000.0         # cooling
        i += n
    return h, c

# ── EnergyPlus: parse .eso into monthly kWh ──────────────────────────────────
def ep_monthly(eso_path,
               hv="Zone Ideal Loads Supply Air Total Heating Energy",
               cv="Zone Ideal Loads Supply Air Total Cooling Energy"):
    var_ids = {}; h_m = np.zeros(12); c_m = np.zeros(12)
    in_dict = True; cur_month = 0
    with open(eso_path) as f:
        for line in f:
            line = line.rstrip()
            if in_dict:
                if line.strip() == "End of Data Dictionary":
                    in_dict = False; continue
                m = re.match(r'^ *(\d+),\d+,.*?,(.+?)\s*\[', line)
                if m:
                    name = m.group(2).strip()
                    if hv.lower() in name.lower():   var_ids[m.group(1)] = "h"
                    elif cv.lower() in name.lower(): var_ids[m.group(1)] = "c"
            else:
                if line.startswith("2,"):            # hourly time-stamp record
                    parts = line.split(",")
                    try: cur_month = int(parts[2].strip().split("/")[0]) - 1
                    except (IndexError, ValueError): pass
                    continue
                parts = line.split(",", 1)
                if len(parts) == 2 and parts[0].strip() in var_ids:
                    try:
                        v = float(parts[1]) / 3_600_000.0   # J → kWh
                        if var_ids[parts[0].strip()] == "h": h_m[cur_month] += v
                        else:                                c_m[cur_month] += v
                    except ValueError: pass
    return h_m, c_m

# ── Compute the four monthly series ──────────────────────────────────────────
iso_h, iso_c = iso_monthly(Q_HC_original)         # ISO 52016 (original engine)
ep_h,  ep_c  = ep_monthly(ESO_PATH)               # EnergyPlus 24.1.0

r = lambda a: [round(float(x), 2) for x in a]

# ── Build the chart ──────────────────────────────────────────────────────────
bar = (
    Bar(init_opts=opts.InitOpts(width="1000px", height="520px",
                                theme=ThemeType.LIGHT))
    .add_xaxis(MONTHS)
    .add_yaxis("Q_H",            r(iso_h), itemstyle_opts=opts.ItemStyleOpts(color="#c0392b"))
    .add_yaxis("Q_H EnergyPlus", r(ep_h),  itemstyle_opts=opts.ItemStyleOpts(color="#e0a30c"))
    .add_yaxis("Q_C",            r(iso_c), itemstyle_opts=opts.ItemStyleOpts(color="#3498db"))
    .add_yaxis("Q_C EnergyPlus", r(ep_c),  itemstyle_opts=opts.ItemStyleOpts(color="#27924a"))
    .set_series_opts(label_opts=opts.LabelOpts(position="top", font_size=9))
    .set_global_opts(
        title_opts=opts.TitleOpts(title="Monthly consumption"),
        yaxis_opts=opts.AxisOpts(name="Energy [kWh]"),
        xaxis_opts=opts.AxisOpts(name="Month"),
        tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="shadow"),
        legend_opts=opts.LegendOpts(pos_top="2%"),
        datazoom_opts=[opts.DataZoomOpts(type_="slider", range_start=0, range_end=100)],
        toolbox_opts=opts.ToolboxOpts(
            feature=opts.ToolBoxFeatureOpts(
                save_as_image=opts.ToolBoxFeatureSaveAsImageOpts(title="Save"),
                restore=opts.ToolBoxFeatureRestoreOpts(title="Restore"),
                data_view=opts.ToolBoxFeatureDataViewOpts(title="Data", lang=["Data view","Close","Refresh"]),
                magic_type=opts.ToolBoxFeatureMagicTypeOpts(line_title="Line", bar_title="Bar"),
            )
        ),
    )
)

# Console summary
print(f"{'Month':>5} | {'Q_H':>6} {'Q_H_EP':>7} | {'Q_C':>6} {'Q_C_EP':>7}")
for i, mo in enumerate(MONTHS):
    print(f"{mo:>5} | {iso_h[i]:6.2f} {ep_h[i]:7.2f} | {iso_c[i]:6.2f} {ep_c[i]:7.2f}")
print(f"{'TOTAL':>5} | {iso_h.sum():6.1f} {ep_h.sum():7.1f} | {iso_c.sum():6.1f} {ep_c.sum():7.1f}  (kWh/yr)")

bar.render_notebook()

## Full Parameter Report\n\nSide-by-side record of every building and simulation parameter across all three engines. The first table lists inputs that are **identical** across engines. The second table lists **engine-specific** measurement parameters that differ by design or methodology.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FULL PARAMETER REPORT — Apt 305, 50 Barry St, Carlton, Melbourne
# Compares all three engines side-by-side; highlights engine-specific fields
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

bui = build_bui()
bp  = bui["building_parameters"]
ig  = {g["name"]: g for g in bp["internal_gains"]}

# ── Shared inputs (identical across all three engines) ───────────────────────
shared = [
    # Location & geometry
    ("LOCATION", "Latitude",                    "-37.800 °",          "-37.800 °",           "-37.800 °"),
    ("LOCATION", "Longitude",                   "144.968 °",          "144.968 °",           "144.968 °"),
    ("LOCATION", "Weather source",              "PVGIS TMY",          "PVGIS TMY",           "PVGIS TMY EPW (same data)"),
    ("GEOMETRY", "Floor area",                  "20 m²",              "20 m²",               "20 m²"),
    ("GEOMETRY", "Volume",                      "54 m³",              "54 m³",               "54 m³"),
    ("GEOMETRY", "Ceiling height",              "2.7 m",              "2.7 m",               "2.7 m"),
    ("GEOMETRY", "Adjacent zones",              "5",                  "5",                   "5 (OtherSideCoeff at 21 °C)"),
    # Envelope
    ("ENVELOPE", "Ext wall U-value",            "1.00 W/m²K",         "1.00 W/m²K",          "R=0.8300 m²K/W → U≈1.00 W/m²K"),
    ("ENVELOPE", "Ext wall solar absorptance",  "0.75",               "0.75",                "0.75 (SolarAbsorptance)"),
    ("ENVELOPE", "Int wall U-value",            "2.50 W/m²K",         "2.50 W/m²K",          "R=0.1400 m²K/W → U≈2.50 W/m²K"),
    ("ENVELOPE", "Int slab U-value",            "1.80 W/m²K",         "1.80 W/m²K",          "R=0.2956 m²K/W → U≈1.80 W/m²K"),
    ("ENVELOPE", "Window U-value",              "5.40 W/m²K",         "5.40 W/m²K",          "5.40 W/m²K (SimpleGlazing)"),
    ("ENVELOPE", "Window SHGC (g-value)",       "0.65",               "0.65",                "0.65"),
    ("ENVELOPE", "Window area",                 "1.62 m² (2×0.81)",   "1.62 m² (2×0.81)",    "1.62 m² (2×0.81)"),
    ("ENVELOPE", "Window orientation",          "West (270°)",        "West (270°)",         "West (270°)"),
    ("ENVELOPE", "Overhang projection",         "0.05 m",             "0.05 m",              "not modelled (EP limitation)"),
    # Thermal mass — zeroed for this comparison
    ("THERMAL MASS", "C_EXT_WALL",              "0 J/m²K",            "0 J/m²K",             "NoMass (0 by definition)"),
    ("THERMAL MASS", "C_INT_WALL",              "0 J/m²K",            "0 J/m²K",             "NoMass (0 by definition)"),
    ("THERMAL MASS", "C_INT_SLAB",              "0 J/m²K",            "0 J/m²K",             "NoMass (0 by definition)"),
    ("THERMAL MASS", "C_WINDOW",                "0 J/m²K",            "0 J/m²K",             "SimpleGlazingSystem (0)"),
    # Setpoints
    ("SETPOINTS", "Heating setpoint",           "18 °C",              "18 °C (overridden)",  "18 °C"),
    ("SETPOINTS", "Heating setback",            "15 °C",              "15 °C (overridden)",  "15 °C"),
    ("SETPOINTS", "Cooling setpoint",           "26 °C",              "26 °C (overridden)",  "26 °C"),
    ("SETPOINTS", "Cooling setback",            "28 °C",              "28 °C (overridden)",  "28 °C"),
    # Occupancy & internal gains
    ("INTERNAL GAINS", "Occupants (peak)",      "8.0 W/m² = 2 people","8.0 W/m² = 2 people", "0.1 p/m² × 160 W/p = 2 people"),
    ("INTERNAL GAINS", "Appliances (peak)",     "5.0 W/m²",           "5.0 W/m²",            "5.0 W/m²"),
    ("INTERNAL GAINS", "Lighting (peak)",       "3.0 W/m²",           "3.0 W/m²",            "3.0 W/m²"),
    # Ventilation
    ("VENTILATION", "Ventilation rate",         "2.0 l/(s·m²)",       "2.0 l/(s·m²)",        "0.002 m³/(s·m²)"),
    # HVAC
    ("HVAC", "System type",                     "Ideal unlimited",    "Ideal unlimited",     "IdealLoadsAirSystem"),
    ("HVAC", "Heating capacity",                "10 MW (unlimited)",  "10 MW (unlimited)",   "NoLimit"),
    ("HVAC", "Cooling capacity",                "10 MW (unlimited)",  "10 MW (unlimited)",   "NoLimit"),
]

# ── Engine-specific measurement parameters ────────────────────────────────────
engine_specific = [
    # Simulation method
    ("METHOD", "Simulation core",               "ISO 52016-1 RC-network (5R1C)",
                                                "ISO 52016-1 RC-network (5R1C)",
                                                "Full heat-balance CTF (EnergyPlus)"),
    ("METHOD", "Time step",                     "Hourly",             "Hourly",              "10-min (Timestep,6)"),
    ("METHOD", "Thermal bridge factor",         "1.5 W/mK",           "1.5 W/mK",            "not modelled"),
    # Latent / humidity
    ("OUTPUTS", "Latent heat calc.",            "No (Q_HC only)",     "Yes (Q_Latent + x_air_in)", "Inherent in full heat balance"),
    ("OUTPUTS", "Humidity balance",             "No",                 "Yes (x_air_in)",      "Inherent"),
    ("OUTPUTS", "DHW integration",              "Separate function",  "Integrated",          "Not modelled"),
    ("OUTPUTS", "Sankey flow data",             "No",                 "Yes (3rd return)",    "Not modelled"),
    # Adjacent zone treatment
    ("ADJ ZONES", "ISO 13789 b_ztu factor",     "Computed",           "Computed",            "Not applicable (fixed T)"),
    ("ADJ ZONES", "Adjacent zone temperature",  "Computed (ISO 13789)","Computed (ISO 13789)","Fixed 21 °C"),
    # ISO 13370 ground coupling
    ("ENVELOPE", "ISO 13370 correction",        "Not applied",        "Skipped (guard added)","Not applicable"),
    # NCC setpoint auto-derivation
    ("SETPOINTS", "NCC auto-derivation",        "Not available",      "Available (disabled for fair comparison)",
                                                "Not applicable"),
    # Weather
    ("WEATHER", "Source / format",              "PVGIS API → DataFrame","PVGIS API → DataFrame","PVGIS API → EPW file"),
    ("WEATHER", "Fetch mechanism",              "Internal (WEATHER_SOURCE=pvgis)", "Internal (WEATHER_SOURCE=pvgis)", "wget → /content/*.epw"),
]

cols = ["Category", "Parameter", "Engine 1: Original pyBuildingEnergy", "Engine 2: AIB Fork", "Engine 3: EnergyPlus"]
shared_df        = pd.DataFrame(shared,        columns=cols)
engine_spec_df   = pd.DataFrame(engine_specific, columns=cols)

print("=" * 100)
print("SHARED BUILDING INPUTS — identical across all three engines")
print("=" * 100)
display(shared_df.set_index(["Category", "Parameter"]))

print()
print("=" * 100)
print("ENGINE-SPECIFIC MEASUREMENT PARAMETERS — where engines differ")
print("=" * 100)
display(engine_spec_df.set_index(["Category", "Parameter"]))
